# AI-Powered Manga Translation Pipeline (Google Colab T4 Runner)

This notebook orchestrates the stage-batched translation pipeline combining **Magi v2** (detection & visual speaker diarization), **manga-ocr** (Japanese optical character recognition), **Rolling Context Memory** (solving Japanese pro-drop), and **Gemini 2.5 Flash** structured translation.

### Hardware Requirement:
- Runtime: **GPU (T4 recommended)**. Check via *Runtime -> Change runtime type -> T4 GPU*.

## Cell 1: Environment Setup & Dependency Installation
Clones the repository, installs all pinned dependencies, and mounts Google Drive for persistent artifact storage.

In [ ]:
# 1. Mount Google Drive for persistent results
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repository if not already in workspace
import os
if not os.path.exists('/content/manga-translation-pipeline'):
    !git clone https://github.com/mshaiel/manga-translation-pipeline.git /content/manga-translation-pipeline

%cd /content/manga-translation-pipeline

# 3. Install core dependencies
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/ragavsachdeva/magi.git

print("✅ Setup complete! Dependencies installed.")

## Cell 2: Configuration & API Secrets
Configure your Gemini API key using Google Colab's built-in Secrets tab (the key icon on the left sidebar) and specify input manga folders.

In [ ]:
import os
from google.colab import userdata

# Retrieve Gemini API key from Colab Secrets
try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    print("✅ Loaded GEMINI_API_KEY from Colab Secrets.")
except Exception:
    print("⚠️ GEMINI_API_KEY not found in Secrets. Please set it via os.environ['GEMINI_API_KEY'] = 'your_key'")

# Define Input and Persistent Output paths
INPUT_PAGES_DIR = "examples/input/pages"
CHARACTER_BANK_DIR = "examples/input/character_bank" # optional
USER_CONTEXT_FILE = "examples/input/user_context.txt" # optional

# Persistent Google Drive export destination
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/manga_translation_output"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print(f"Target Output Directory: {DRIVE_OUTPUT_DIR}")

## Cell 3: Run Stage-Batched Translation Pipeline
Executes the full pipeline: Magi v2 -> VRAM release -> manga-ocr -> VRAM release -> Rolling Context -> Gemini 2.5 Flash -> Pillow typesetting -> PDF/CBZ export.

In [ ]:
from pathlib import Path
from src.pipeline import MangaTranslationPipeline

# Collect input manga page images sorted by filename
valid_extensions = {'.png', '.jpg', '.jpeg', '.webp'}
page_files = sorted([
    p for p in Path(INPUT_PAGES_DIR).iterdir()
    if p.suffix.lower() in valid_extensions
])
print(f"Found {len(page_files)} manga pages to translate.")

# Initialize pipeline with Google Drive storage
pipeline = MangaTranslationPipeline(
    config={
        "pipeline": {
            "output_dir": DRIVE_OUTPUT_DIR,
            "checkpoints_dir": f"{DRIVE_OUTPUT_DIR}/checkpoints",
            "enable_checkpoints": True,
        },
        "context": {
            "sliding_window_pages": 6,
        }
    }
)

# Run stage-batched translation
results = pipeline.run(
    chapter_pages=page_files,
    chapter_name="chapter_01",
    user_context=USER_CONTEXT_FILE if Path(USER_CONTEXT_FILE).exists() else None,
    resume_from_checkpoints=True,
)

print("🎉 Pipeline execution finished successfully!")

## Cell 4: Visual Review & Quality Assurance Audit
Display side-by-side comparisons of the original Japanese and translated English pages, and inspect the `quality_report.json` audit summary.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Display first page comparison
if page_files and results.get("translated_images"):
    orig = Image.open(page_files[0])
    trans = results["translated_images"][0]

    fig, axes = plt.subplots(1, 2, figsize=(16, 12))
    axes[0].imshow(orig)
    axes[0].set_title("Original Japanese", fontsize=14)
    axes[0].axis('off')

    axes[1].imshow(trans)
    axes[1].set_title("Translated English (Typeset)", fontsize=14)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

# Print Quality & Cost Summary
rep = results["quality_report"]
print("=" * 60)
print(f"Chapter: {rep.chapter} | Total Pages: {rep.total_pages}")
print(f"Total Dialogue Bubbles: {rep.total_text_boxes} | Total SFX: {rep.total_sfx}")
print(f"Estimated API Cost: ${rep.pipeline_metadata.total_api_cost_estimate_usd:.4f} USD")
print(f"PDF Generated: {results['pdf_path']}")
print(f"CBZ Generated: {results['cbz_path']}")
print("=" * 60)

## Cell 5: Launch Interactive Gradio Web App
Launches the browser-based demo with a shareable public link (`share=True`) allowing direct drag-and-drop page translation and live side-by-side previews.

In [ ]:
from src.ui.app import create_ui

app = create_ui(pipeline_instance=pipeline)
app.launch(share=True, debug=False)